# Experiment 035: matched LoRA versus full GRPO (Colab)

The zero-contact pilot selected 3B. This session runs the single allocator-only rung A retry before the remaining matched pairs.

## 0. Compute units and Drive

In [ ]:
import glob, json, os, time
ALLOCATOR_CONFIG = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ALLOCATOR_CONFIG
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 491.95
unit_rate = 6.77
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

old_sessions = [path for path in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not path.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None or unit_rate is None:
    print('fill cu_before and unit_rate, then rerun this cell')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])
    if not os.path.exists(open_path):
        record = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                  'notebook': '035_colab_matched_grpo', 'colab_cu_before': cu_before,
                  'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(record, f, indent=2)
        print('saved open session:', session_id)

session 20260817_175921 | cu_before = 491.95
Mounted at /content/drive
saved open session: 20260817_175921


## 1. Runtime

In [2]:
import shutil, subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(gpu.strip())
print(f'total vram: {total_gib:.1f} GiB')
assert 'A100' in gpu and total_gib >= 79, 'need the A100 80GB runtime'
print('local free:', round(shutil.disk_usage('/content').free / 1024**3, 1), 'GiB')
print('drive free:', round(shutil.disk_usage(ROOT).free / 1024**3, 1), 'GiB')

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-2604a012-eedd-743c-8eb8-2ca4c98ffe15)
total vram: 79.3 GiB
local free: 188.6 GiB
drive free: 179.2 GiB


## 2. Repository and dependencies

In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

working in: /content/INM363-IndividualProject-Dissertation/001_Code


In [4]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trl==1.7.1', 'transformers==5.5.4', 'peft==0.19.1',
                'accelerate==1.14.0', 'datasets==4.8.4', 'wandb', 'pyyaml'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=True)
print('dependencies installed')

dependencies installed


In [5]:
import getpass, os
wandb_key = None
try:
    from google.colab import userdata
    wandb_key = userdata.get('WANDB_API_KEY')
except Exception:
    wandb_key = None
if not wandb_key:
    wandb_key = getpass.getpass('W&B API key: ').strip()
assert wandb_key, 'WANDB_API_KEY is required'
os.environ['WANDB_API_KEY'] = wandb_key
wandb_key = None
print('W&B key loaded')

W&B key loaded


## 3. Wiring checks

In [6]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_035_matched_grpo.py',
    'tests/test_training_wrapper_contracts.py',
    'tests/test_checkpoint.py',
    'tests/test_sentinel.py',
    'tests/test_protocol_freeze.py',
], check=True)
print('wiring checks passed')

wiring checks passed


In [7]:
import json, os, shutil, sys, time
sys.path.insert(0, os.getcwd())

from config import load_config
from file_utils import read_json
from scripts.grpo_worker import stream_subprocess
from scripts.s35_matched_grpo import (
    ALLOCATOR_CONFIG, RUNGS_035, TRAINING_COMPLETE, arm_tag, bind_start,
    commit_endpoint,
    commit_execution_lock, commit_execution_override, commit_pair_parity,
    commit_scale_selection, commit_start_probe, config_path, endpoint_state,
    prepare_allocator_retry, safe_close_checkpoints,
)

TECH = os.path.join(ROOT, 'results', '035_technical')
os.makedirs(TECH, exist_ok=True)
start_records = {}
for scale in ('B3', 'B15'):
    for seed in (0, 1, 2):
        cfg = load_config(config_path(scale, seed, 'full'))
        path = os.path.join(TECH, 'starts', scale, f'seed{seed}.json')
        start_records[(scale, seed)] = bind_start(cfg, path)
print('all six starts verified')

all six starts verified


## 4. Select the session

In [8]:
ACTION = 'science_seed0'  # pilot_3b, pilot_b15, science_seed0 or science
SCIENCE_START_RUNG = 'A'
assert ACTION in ('pilot_3b', 'pilot_b15', 'science_seed0', 'science')
if ACTION.startswith('pilot_'):
    assert shutil.disk_usage('/content').free / 1024**3 >= 80, 'need at least 80 GiB free locally'
if ACTION.startswith('science'):
    selection = read_json(os.path.join(TECH, '035_scale_selection.json'))
    override = commit_execution_override(ROOT, selection, SCIENCE_START_RUNG)
    retry = prepare_allocator_retry(ROOT, selection, override)
    assert os.environ.get('PYTORCH_CUDA_ALLOC_CONF') == retry['allocator_config'] == ALLOCATOR_CONFIG
    report = read_json(os.path.join(TECH, 'pilots', selection['selected_scale'],
                                    selection['pilot_rung'], 'pilot_report.json'))
    checkpoint_gib = report['checkpoint_bytes'] / 1024**3
    required_gib = 3 * checkpoint_gib + (50 if selection['selected_scale'] == 'B3' else 30)
    print(f'science Drive gate: {required_gib:.1f} GiB')
    print('science rung override:', override['requested_rung'])
    print('allocator retry:', retry['allocator_config'])
    assert shutil.disk_usage(ROOT).free / 1024**3 >= required_gib, 'not enough Drive space'
print('action:', ACTION)

science Drive gate: 153.5 GiB
science rung override: A
allocator retry: expandable_segments:True
action: science_seed0


In [9]:
def append_jsonl(path, row):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'a') as f:
        f.write(json.dumps(row) + '\n')

def run_worker(command, status_path, result_dir, label):
    if os.path.exists(status_path):
        os.remove(status_path)
    started = time.time()
    process = stream_subprocess(command)
    if os.path.exists(status_path):
        status = read_json(status_path)
    else:
        status = {'completed': False, 'error_type': 'MissingStatus',
                  'error_message': 'worker exited without a status file'}
    status = {**status, 'label': label, 'returncode': process.returncode,
              'parent_wall_seconds': round(time.time() - started, 1)}
    append_jsonl(os.path.join(result_dir, '035_attempts.jsonl'), status)
    append_jsonl(os.path.join(ROOT, 'results', 'compute_ledger.jsonl'), {
        'run_tag': label, 'condition': f'035_{status.get("task", "worker")}',
        'timings_sec': {'worker': status['parent_wall_seconds']},
        'train_sec': status['parent_wall_seconds'] if status.get('task') in ('pilot', 'train') else 0,
        'train_steps': status.get('completed_updates', 0),
        'completed': bool(status.get('completed')), 'gpu': status.get('gpu'),
    })
    return process, status

def worker_command(task, cfg_path, start_path, rung, status_path, out_dir, result_dir, **kwargs):
    command = [
        sys.executable, 'scripts/s35_worker.py', '--task', task,
        '--config', cfg_path, '--start-record', start_path, '--rung', rung,
        '--status', status_path, '--out-dir', out_dir, '--result-dir', result_dir,
    ]
    for key, value in kwargs.items():
        if value is not None:
            command.extend(['--' + key.replace('_', '-'), str(value)])
    return command

## 5. Technical pilot

In [10]:
if ACTION.startswith('pilot_'):
    scale = 'B3' if ACTION == 'pilot_3b' else 'B15'
    cfg_path = config_path(scale, 0, 'full')
    start_path = os.path.join(TECH, 'starts', scale, 'seed0.json')
    selected_rung = None
    for rung in RUNGS_035:
        pilot_dir = os.path.join(TECH, 'pilots', scale, rung)
        report_path = os.path.join(pilot_dir, 'pilot_report.json')
        if os.path.exists(report_path):
            report = read_json(report_path)
            if report.get('passed'):
                selected_rung = rung
                break
            continue

        os.makedirs(pilot_dir, exist_ok=True)
        local_out = f'/content/035_pilot_{scale}_{rung}'
        if os.path.isdir(local_out):
            shutil.rmtree(local_out)
        precision_path = os.path.join(pilot_dir, 'precision_report.json')
        status_path = os.path.join(pilot_dir, 'worker_status.json')
        command = worker_command(
            'pilot', cfg_path, start_path, rung, status_path, local_out, pilot_dir,
            pilot_report=report_path, precision_report=precision_path)
        process, status = run_worker(command, status_path, pilot_dir, f'035_pilot_{scale}_{rung}')
        if os.path.isdir(local_out):
            shutil.rmtree(local_out)
        if not os.path.exists(report_path):
            raise RuntimeError(f'pilot {scale} rung {rung} stopped without a technical verdict: {status}')
        report = read_json(report_path)
        if report.get('passed'):
            selected_rung = rung
            break
        margin_only = (status.get('completed') and report.get('peak_reserved_fraction', 1) > 0.95)
        if not status.get('cuda_oom') and not margin_only:
            raise RuntimeError(f'pilot {scale} rung {rung} failed an integrity gate: {report}')

    if selected_rung:
        selection = commit_scale_selection(ROOT)
        print('selected:', selection)
    elif scale == 'B3':
        print('3B ladder exhausted. Change ACTION to pilot_b15 and Run all in a fresh session.')
    else:
        commit_scale_selection(ROOT)

## 6. Matched scientific arms

In [11]:
def start_record_path(scale, seed):
    return os.path.join(TECH, 'starts', scale, f'seed{seed}.json')

def run_start_probe(scale, seed):
    result_dir = os.path.join(ROOT, 'results', f'035_start_{scale}_seed{seed}')
    manifest = os.path.join(result_dir, '035_start_probe_manifest.json')
    if os.path.exists(manifest):
        return commit_start_probe(result_dir, '/content/unused', start_records[(scale, seed)])
    scratch = f'/content/035_start_probe_{scale}_{seed}'
    if os.path.isdir(scratch):
        shutil.rmtree(scratch)
    os.makedirs(scratch)
    status_path = os.path.join(result_dir, 'start_probe_status.json')
    cfg_path = config_path(scale, seed, 'full')
    command = worker_command(
        'eval', cfg_path, start_record_path(scale, seed), 'A', status_path,
        start_records[(scale, seed)]['start'], result_dir,
        branch='start', scratch_dir=scratch)
    process, status = run_worker(command, status_path, result_dir, f'035_start_{scale}_seed{seed}')
    if process.returncode != 0 or not status.get('completed'):
        raise RuntimeError(f'start probe failed for {scale} seed {seed}')
    record = commit_start_probe(result_dir, scratch, start_records[(scale, seed)])
    shutil.rmtree(scratch)
    return record

def run_parity_pair(scale, seed, rung):
    pair_dir = os.path.join(TECH, 'parity', scale, f'seed{seed}', rung)
    pair_path = os.path.join(pair_dir, 'pair_parity.json')
    os.makedirs(pair_dir, exist_ok=True)
    records = {}
    for mode in ('lora', 'full'):
        record_path = os.path.join(pair_dir, f'{mode}_parity.json')
        if not os.path.exists(record_path):
            local_out = f'/content/035_parity_{scale}_{seed}_{rung}_{mode}'
            if os.path.isdir(local_out):
                shutil.rmtree(local_out)
            status_path = os.path.join(pair_dir, f'{mode}_status.json')
            command = worker_command(
                'parity', config_path(scale, seed, mode), start_record_path(scale, seed),
                rung, status_path, local_out, pair_dir, parity_record=record_path)
            process, status = run_worker(
                command, status_path, pair_dir, f'035_parity_{scale}_seed{seed}_{rung}_{mode}')
            if process.returncode != 0 or not status.get('completed'):
                raise RuntimeError(f'parity failed for {scale} seed {seed} {mode}')
            if os.path.isdir(local_out):
                shutil.rmtree(local_out)
        records[mode] = read_json(record_path)
    return commit_pair_parity(pair_path, records['lora'], records['full'])

In [12]:
def training_paths(scale, seed, mode):
    tag = arm_tag(scale, seed, mode)
    return (
        os.path.join(ROOT, 'results', tag),
        os.path.join(ROOT, 'checkpoints', tag, 'grpo'),
    )

def run_training(scale, seed, mode, rung, allow_clean_retry=False):
    rungs = list(RUNGS_035)
    start_i = rungs.index(rung)
    choices = rungs[start_i:] if allow_clean_retry else [rung]
    for current_rung in choices:
        run_start_probe(scale, seed)
        run_parity_pair(scale, seed, current_rung)
        result_dir, output_dir = training_paths(scale, seed, mode)
        cfg = load_config(config_path(scale, seed, mode))
        complete_path = os.path.join(result_dir, TRAINING_COMPLETE)
        precision_path = os.path.join(result_dir, 'precision_report.json')
        if os.path.exists(complete_path):
            from scripts.s35_matched_grpo import commit_training_complete
            return commit_training_complete(
                cfg, result_dir, output_dir, start_records[(scale, seed)],
                current_rung, precision_path)

        os.makedirs(result_dir, exist_ok=True)
        status_path = os.path.join(result_dir, 'worker_status.json')
        command = worker_command(
            'train', config_path(scale, seed, mode), start_record_path(scale, seed),
            current_rung, status_path, output_dir, result_dir,
            precision_report=precision_path)
        command.extend(['--report-to', 'tensorboard', 'wandb'])
        process, status = run_worker(
            command, status_path, result_dir, f'035_{scale}_seed{seed}_{mode}_{current_rung}')
        if process.returncode == 0 and status.get('completed'):
            return read_json(complete_path)
        if not allow_clean_retry or not status.get('cuda_oom'):
            raise RuntimeError(f'training failed for {scale} seed {seed} {mode}')
        failed_root = os.path.join(ROOT, 'failed_attempts', '035',
                                   f'{scale}_seed{seed}_{mode}_{current_rung}')
        os.makedirs(os.path.dirname(failed_root), exist_ok=True)
        if os.path.exists(failed_root):
            raise RuntimeError(f'failed-attempt archive already exists: {failed_root}')
        os.makedirs(failed_root)
        if os.path.isdir(result_dir):
            shutil.move(result_dir, os.path.join(failed_root, 'results'))
        if os.path.isdir(output_dir):
            shutil.move(os.path.dirname(output_dir), os.path.join(failed_root, 'checkpoints'))
    raise RuntimeError(f'all registered rungs failed for {scale} seed {seed} {mode}')

def run_endpoint(scale, seed, mode, training):
    result_dir, output_dir = training_paths(scale, seed, mode)
    state = endpoint_state(result_dir, training)
    if state == 'tampered':
        raise RuntimeError(f'committed endpoint changed for {scale} seed {seed} {mode}')
    if state == 'committed':
        return read_json(os.path.join(result_dir, '035_eval_manifest.json'))
    scratch = f'/content/035_eval_{scale}_{seed}_{mode}'
    if os.path.isdir(scratch):
        shutil.rmtree(scratch)
    os.makedirs(scratch)
    status_path = os.path.join(result_dir, 'eval_status.json')
    command = worker_command(
        'eval', config_path(scale, seed, mode), start_record_path(scale, seed),
        training['rung'], status_path, output_dir, result_dir,
        branch=mode, scratch_dir=scratch)
    process, status = run_worker(
        command, status_path, result_dir, f'035_eval_{scale}_seed{seed}_{mode}')
    if process.returncode != 0 or not status.get('completed'):
        raise RuntimeError(f'endpoint evaluation failed for {scale} seed {seed} {mode}')
    record = commit_endpoint(result_dir, scratch, training)
    shutil.rmtree(scratch)
    safe_close_checkpoints(result_dir, output_dir)
    return record

In [ ]:
if ACTION in ('science_seed0', 'science'):
    selection = read_json(os.path.join(TECH, '035_scale_selection.json'))
    scale = selection['selected_scale']
    override = commit_execution_override(ROOT, selection, SCIENCE_START_RUNG)
    retry = prepare_allocator_retry(ROOT, selection, override)
    assert os.environ.get('PYTORCH_CUDA_ALLOC_CONF') == retry['allocator_config']
    lock_path = os.path.join(TECH, 'execution_lock.json')

    if not os.path.exists(lock_path):
        full0 = run_training(
            scale, 0, 'full', override['requested_rung'], allow_clean_retry=False)
        lock = commit_execution_lock(ROOT, scale, full0['rung'], full0)
        run_endpoint(scale, 0, 'full', full0)
    else:
        lock = read_json(lock_path)
        assert lock['scale'] == scale

    if ACTION == 'science':
        rung = lock['rung']
        for seed in (0, 1, 2):
            run_start_probe(scale, seed)
            run_parity_pair(scale, seed, rung)
            for mode in ('full', 'lora'):
                training = run_training(scale, seed, mode, rung)
                run_endpoint(scale, seed, mode, training)
            print('completed pair:', scale, 'seed', seed)
    else:
        print('seed 0 full arm completed at rung', lock['rung'])

README.md: 100% 53.2k/53.2k [00:00<00:00, 77.8MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 204MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00:00<?, ?B/s]
high_school_psychology/test-00000-of-000(…): downloading bytes: 100% 91.9k/91.9k [00:01<00:00, 68.0kB/s, 8.94kB/s  ]
high_school_psychology/test-00000-of-000(…): reconstructing file: 100% 92.8k/92.8k [00:01<00:00, 68.6kB/s, 9.02kB/s  ]

high_school_psychology/validation-00000-(…): downloading bytes:   0% 0.00/15.2k [00:00<?, ?B/s]
high_school_psychology/validation-00000-(…): downloading bytes: 100% 13.4k/13.4k [00:01<00:00, 10.2kB/s, 1.30kB/s  ]
high_school_psychology/validation-00000-(…): reconstructing file: 100% 15.2k/15.2k [00:01<00:00, 11.6kB/s, 1.48kB/s  ]

high_school_psychology/dev-00000-of-0000(…): downloading bytes:   0% 0.00/5.18k [00:00<?, ?B/s]
high_school_psychology/dev-00000-of-0000(…): downloading bytes: 100% 3.21k/3.21k [00:01<00:00, 2.61kB/s,   313B/s  ]
high_schoo

RuntimeError: training failed for B3 seed 0 full

## 7. Current state

In [ ]:
selection_path = os.path.join(TECH, '035_scale_selection.json')
print('selection:', read_json(selection_path) if os.path.exists(selection_path) else 'not selected')
for path in sorted(glob.glob(os.path.join(TECH, 'pilots', '*', '*', 'pilot_report.json'))):
    row = read_json(path)
    print(path.replace(ROOT + '/', ''), '| passed', row.get('passed'),
          '| peak', round(row.get('peak_reserved_fraction', 0), 3))
for path in sorted(glob.glob(os.path.join(ROOT, 'results', '035_*', TRAINING_COMPLETE))):
    row = read_json(path)
    print(os.path.basename(os.path.dirname(path)), row['parameter_mode'],
          row['seed'], row['rung'], row['completed'])

## 8. Close compute units

In [ ]:
cu_after = None
print('cu_after =', cu_after if cu_after is not None else 'FILL ME')
if cu_after is None:
    print('fill cu_after, then rerun this cell')
elif not os.path.exists(open_path):
    print('no open session record')
else:
    with open(open_path) as f:
        record = json.load(f)
    record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
    record['colab_cu_after'] = cu_after
    record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
    record['completed'] = True
    with open(UNITS_LOG, 'a') as f:
        f.write(json.dumps(record) + '\n')
    os.remove(open_path)
    print('logged:', record)